# Pandas Masterclass: From Beginner to Practical Data Analysis

**Goal:** Learn the most useful pandas skills for data analysis, data cleaning, and preparing data for machine learning.

## Practice dataset
We'll use the public **Tips** dataset (restaurant bills and tips):
- **CSV URL:** https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv
- **Dataset columns:** `total_bill`, `tip`, `sex`, `smoker`, `day`, `time`, `size`
- **Source repository:** https://github.com/mwaskom/seaborn-data

The notebook downloads the CSV directly. If you are offline, download the CSV from the URL and update `DATA_URL` to the local file path.

## How to use this notebook
1. Run cells from top to bottom.
2. Read the explanation before running the examples.
3. Try the **Practice task** cells yourself before opening the answer cells.
4. Use this notebook as a reference; pandas is large, so this covers the core skills you'll use most often rather than every function in the library.

**Requirements:** Python, pandas, NumPy, and Matplotlib. Install missing packages with `pip install pandas numpy matplotlib`.

## 1. Import libraries and load data

`pandas` is usually imported as `pd`. A DataFrame is a table; a Series is one column or one-dimensional labelled data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
df = pd.read_csv(DATA_URL)

print("Pandas version:", pd.__version__)
print("Dataset shape:", df.shape)
df.head()

## 2. Understand the DataFrame

Start every unfamiliar dataset by checking its size, column names, data types, and a few rows.

In [ ]:
df.head()          # first 5 rows

In [ ]:
df.tail(3)         # last 3 rows

In [ ]:
df.sample(5, random_state=42)  # random sample; fixed seed makes it repeatable

In [ ]:
df.shape           # (rows, columns)

In [ ]:
df.columns         # column labels

In [ ]:
df.index           # row index

In [ ]:
df.dtypes          # data type of each column

In [ ]:
df.info()          # concise structure, non-null counts, memory

In [ ]:
df.describe()      # numeric summary statistics

In [ ]:
df.describe(include="all")  # include categorical columns too

## 3. Core concepts: Series and DataFrame

A Series is one-dimensional. A DataFrame is a two-dimensional table made of aligned Series.

In [ ]:
type(df), type(df["tip"]), df["tip"].head()

In [ ]:
# Create a small DataFrame from a dictionary
mini = pd.DataFrame({
    "name": ["Ama", "Kojo", "Esi"],
    "age": [22, 25, 21],
    "city": ["Accra", "Kumasi", "Ho"]
})
mini

## 4. Select columns and rows

- `df['tip']`: one column as a Series.
- `df[['tip', 'day']]`: multiple columns as a DataFrame.
- `.loc`: label-based selection.
- `.iloc`: position-based selection.

In [ ]:
df["tip"].head()

In [ ]:
df[["total_bill", "tip", "day"]].head()

In [ ]:
df.loc[0:4, ["total_bill", "tip"]]  # labels; end label is included for this range

In [ ]:
df.iloc[0:5, 0:3]  # positions; end position is excluded

In [ ]:
df.at[0, "tip"]     # fast scalar access by label

In [ ]:
df.iat[0, 1]        # fast scalar access by position

## 5. Filter rows with conditions

Use comparison operators and wrap each condition in parentheses when combining conditions with `&` (AND) or `|` (OR). Use `~` for NOT.

In [ ]:
# Bills greater than 30
df[df["total_bill"] > 30].head()

In [ ]:
# Bills above 30 AND tips above 4
df[(df["total_bill"] > 30) & (df["tip"] > 4)].head()

In [ ]:
# Saturday OR Sunday
df[df["day"].isin(["Sat", "Sun"])].head()

In [ ]:
# Exclude smokers
df[~(df["smoker"] == "Yes")].head()

In [ ]:
# Query syntax can be readable for simple filters
df.query("total_bill > 30 and tip > 4").head()

## 6. Sort data and rank values

`sort_values` sorts by one or more columns. `ascending=False` puts the largest values first.

In [ ]:
df.sort_values("tip", ascending=False).head(10)

In [ ]:
df.sort_values(["day", "total_bill"], ascending=[True, False]).head(10)

In [ ]:
df["tip_rank"] = df["tip"].rank(method="dense", ascending=False)
df[["tip", "tip_rank"]].head()

In [ ]:
# Remove the teaching column so later examples use the original dataset
df = df.drop(columns=["tip_rank"])

## 7. Create and transform columns

Pandas operations are vectorised: apply them to whole columns rather than looping through every row.

In [ ]:
df["tip_pct"] = (df["tip"] / df["total_bill"]) * 100
df["bill_per_person"] = df["total_bill"] / df["size"]
df[["total_bill", "tip", "tip_pct", "bill_per_person"]].head()

In [ ]:
# Conditional values with np.where
df["large_bill"] = np.where(df["total_bill"] >= 30, "Yes", "No")
df[["total_bill", "large_bill"]].head()

In [ ]:
# Multiple conditions with np.select
conditions = [df["total_bill"] < 15, df["total_bill"] < 30]
choices = ["Small", "Medium"]
df["bill_group"] = np.select(conditions, choices, default="Large")
df[["total_bill", "bill_group"]].head()

In [ ]:
# Apply a function to a column (vectorised expressions are usually preferred)
df["tip_rounded"] = df["tip"].round(1)
df[["tip", "tip_rounded"]].head()

## 8. Rename, add, drop, and reorder columns

In [ ]:
df.rename(columns={"size": "party_size"}, inplace=True)
df[["party_size", "total_bill", "tip"]].head()

In [ ]:
# Add a column
df["service_note"] = "Restaurant visit"

# Drop columns by name; axis=1 means columns
df = df.drop(columns=["service_note"])

In [ ]:
# Reorder columns
df = df[["total_bill", "tip", "tip_pct", "bill_per_person",
         "sex", "smoker", "day", "time", "party_size",
         "large_bill", "bill_group", "tip_rounded"]]
df.head()

## 9. Missing values

Missing data may appear as `NaN`, `None`, or `pd.NA`. Check missingness before deciding whether to remove or fill values.

In [ ]:
df.isna().sum()                  # missing count per column

In [ ]:
df.isna().mean().mul(100).round(2)  # percent missing per column

In [ ]:
df.notna().sum()                 # non-missing count per column

In [ ]:
# Examples of common strategies (these examples use a copy)
df_missing_demo = df.copy()
df_missing_demo.loc[0, "tip"] = np.nan
df_missing_demo["tip"].isna().sum()

In [ ]:
# Drop rows with missing values in selected columns
df_missing_demo.dropna(subset=["tip"]).head()

In [ ]:
# Fill missing numeric values with the median (example only)
df_missing_demo["tip"] = df_missing_demo["tip"].fillna(df_missing_demo["tip"].median())
df_missing_demo["tip"].isna().sum()

**Choosing a strategy:** dropping rows is simple but loses data; mean/median imputation can distort distributions; categorical values may be filled with a meaningful label such as `"Unknown"`. In machine learning, fit imputers on training data only to avoid data leakage.

## 10. Duplicate rows

In [ ]:
df.duplicated().sum()        # number of duplicate rows

In [ ]:
df.drop_duplicates().shape   # preview resulting shape without changing df

In [ ]:
# Keep first occurrence by default; subset can identify duplicates by selected columns
df.drop_duplicates(subset=["total_bill", "tip", "day"]).head()

## 11. Data types and conversion

Correct data types improve correctness and can reduce memory use.

In [ ]:
df.dtypes

In [ ]:
# Numeric conversion; invalid values become missing
pd.to_numeric(pd.Series(["10", "20", "unknown"]), errors="coerce")

In [ ]:
# String cleanup example
pd.Series(["  Accra ", "KUMASI", None]).str.strip().str.title()

In [ ]:
# Convert a column to category when it has a limited set of repeated labels
df["day"] = df["day"].astype("category")
df["day"].dtype

## 12. String operations

Use `.str` methods for text columns. Many operations work across a whole Series.

In [ ]:
names = pd.Series(["  Ama Mensah", "KOJO BOATENG ", "esi"])
names.str.strip()

In [ ]:
names.str.strip().str.title()

In [ ]:
names.str.lower().str.contains("ama", na=False)

In [ ]:
pd.Series(["Ama Mensah", "Kojo Boateng"]).str.split(" ", expand=True)

In [ ]:
pd.Series(["Order-102", "Order-205"]).str.extract(r"(\d+)")

In [ ]:
pd.Series(["GH₵ 12.50", "GH₵ 8.00"]).str.replace(r"[^0-9.]", "", regex=True).astype(float)

## 13. GroupBy: split, apply, combine

`groupby` is one of the most important pandas tools. It groups rows, calculates summaries, and returns a compact result.

In [ ]:
df.groupby("day", observed=False)["total_bill"].mean().sort_values(ascending=False)

In [ ]:
df.groupby(["day", "time"], observed=False).agg(
    rows=("tip", "size"),
    average_bill=("total_bill", "mean"),
    median_tip=("tip", "median"),
    total_tips=("tip", "sum")
).round(2)

In [ ]:
# Multiple summaries by group
df.groupby("smoker", observed=False).agg({
    "total_bill": ["mean", "median", "max"],
    "tip": ["mean", "sum"]
}).round(2)

In [ ]:
# Transform returns a result aligned to the original rows
df["day_avg_bill"] = df.groupby("day", observed=False)["total_bill"].transform("mean")
df[["day", "total_bill", "day_avg_bill"]].head()

In [ ]:
# Filter entire groups based on a group-level condition
df.groupby("day", observed=False).filter(lambda group: len(group) > 20)["day"].value_counts()

## 14. Pivot tables and reshaping

Use `pivot_table` for spreadsheet-style summaries, `melt` to turn wide data into long data, and `pivot` to reshape without aggregation when keys are unique.

In [ ]:
pd.pivot_table(
    df, index="day", columns="time", values="total_bill",
    aggfunc="mean", observed=False
).round(2)

In [ ]:
# Crosstab: frequency table
pd.crosstab(df["day"], df["time"])

In [ ]:
# Melt a tiny wide table into long form
wide = pd.DataFrame({"person": ["A", "B"], "math": [80, 90], "science": [85, 88]})
long = wide.melt(id_vars="person", var_name="subject", value_name="score")
long

In [ ]:
# Pivot back; each person-subject combination must be unique
long.pivot(index="person", columns="subject", values="score")

In [ ]:
# Cut a numeric variable into bins
pd.cut(df["total_bill"], bins=[0, 15, 30, np.inf], labels=["Small", "Medium", "Large"]).value_counts()

## 15. Combine datasets: concat, merge, join

- `concat` stacks tables vertically or horizontally.
- `merge` joins using key columns, like SQL joins.
- `join` commonly joins on index.

In [ ]:
left = pd.DataFrame({"customer_id": [1, 2, 3], "name": ["Ama", "Kojo", "Esi"]})
right = pd.DataFrame({"customer_id": [1, 2, 4], "spend": [100, 150, 80]})
left.merge(right, on="customer_id", how="inner")

In [ ]:
left.merge(right, on="customer_id", how="left")  # preserve all rows from left

In [ ]:
left.merge(right, on="customer_id", how="outer", indicator=True)

In [ ]:
pd.concat([left, left], ignore_index=True)  # stack rows

**Join types:** `inner` keeps matching keys; `left` keeps all left keys; `right` keeps all right keys; `outer` keeps all keys from both. Check key uniqueness before merging to avoid accidental row multiplication.

In [ ]:
# Check key uniqueness
left["customer_id"].is_unique, right["customer_id"].is_unique

## 16. Dates and time series

Convert strings to datetime with `pd.to_datetime`. Use the `.dt` accessor for year/month/day components.

In [ ]:
dates = pd.Series(["2025-01-05", "2025-02-14", "not-a-date"])
pd.to_datetime(dates, errors="coerce")

In [ ]:
dates = pd.to_datetime(pd.Series(["2025-01-05", "2025-02-14", "2025-03-20"]))
dates.dt.year, dates.dt.month, dates.dt.day_name()

In [ ]:
date_df = pd.DataFrame({
    "date": pd.date_range("2025-01-01", periods=6, freq="D"),
    "sales": [10, 12, 9, 15, 20, 18]
}).set_index("date")
date_df

In [ ]:
date_df.resample("W")["sales"].sum()  # weekly sum

In [ ]:
date_df["rolling_3_day_avg"] = date_df["sales"].rolling(3, min_periods=1).mean()
date_df

In [ ]:
date_df["previous_day_sales"] = date_df["sales"].shift(1)
date_df["daily_change"] = date_df["sales"].diff()
date_df

## 17. Basic statistics and correlations

In [ ]:
df[["total_bill", "tip", "tip_pct", "party_size"]].mean()

In [ ]:
df[["total_bill", "tip", "tip_pct", "party_size"]].median()

In [ ]:
df[["total_bill", "tip", "tip_pct", "party_size"]].quantile([0.25, 0.5, 0.75])

In [ ]:
df[["total_bill", "tip", "tip_pct", "party_size"]].corr(numeric_only=True)

In [ ]:
df["day"].value_counts()

In [ ]:
df["day"].value_counts(normalize=True).mul(100).round(1)  # percentages

## 18. Plotting with pandas

Pandas plotting uses Matplotlib. Plots can help identify patterns, outliers, and relationships.

In [ ]:
df["total_bill"].plot(kind="hist", bins=20, title="Distribution of total bills")
plt.xlabel("Total bill")
plt.show()

In [ ]:
df.groupby("day", observed=False)["tip"].mean().plot(kind="bar", title="Average tip by day")
plt.ylabel("Average tip")
plt.show()

In [ ]:
df.plot(kind="scatter", x="total_bill", y="tip", title="Bill versus tip")
plt.show()

In [ ]:
df[["total_bill", "tip", "tip_pct"]].plot(kind="box", title="Numeric columns: boxplots")
plt.show()

## 19. Read and write files

Pandas supports CSV, Excel, JSON, Parquet, SQL and more. Optional packages may be needed for some formats.

In [ ]:
# CSV
df.to_csv("tips_cleaned.csv", index=False)
# Read it back
df_check = pd.read_csv("tips_cleaned.csv")
df_check.head()

In [ ]:
# Excel examples (requires openpyxl or another supported Excel engine)
# df.to_excel("tips_cleaned.xlsx", index=False)
# df_excel = pd.read_excel("tips_cleaned.xlsx")

In [ ]:
# JSON examples
df.head().to_json("tips_sample.json", orient="records", indent=2)

In [ ]:
# Parquet examples (requires pyarrow or fastparquet)
# df.to_parquet("tips.parquet", index=False)
# df_parquet = pd.read_parquet("tips.parquet")

## 20. Indexes and resetting the index

In [ ]:
df_indexed = df.set_index("day")
df_indexed.head()

In [ ]:
df_indexed.reset_index().head()

In [ ]:
# Sort by index
df_indexed.sort_index().head()

## 21. Efficient pandas habits

1. Prefer vectorised operations over Python loops.
2. Select only columns you need when working with large files.
3. Use `usecols=` and `dtype=` in `read_csv` when appropriate.
4. Avoid chained assignment; use `.loc` for explicit updates.
5. Inspect `.shape`, `.dtypes`, and missing values after major transformations.
6. Use `category` for repeated labels when useful, but check whether it helps.
7. For large datasets, consider chunked CSV reading with `chunksize=`.

In [ ]:
# Recommended explicit update pattern
df.loc[df["total_bill"] > 30, "large_bill"] = "Yes"
df.loc[df["total_bill"] <= 30, "large_bill"] = "No"
df["large_bill"].value_counts()

In [ ]:
# Read a CSV in chunks (example; processes each chunk separately)
# for chunk in pd.read_csv(DATA_URL, chunksize=1000):
#     print(chunk.shape)

In [ ]:
# Memory usage
df.memory_usage(deep=True).sort_values(ascending=False)

## 22. Common errors and how to fix them

- **`KeyError: 'column'`:** inspect `df.columns`; names may have spaces or different capitalisation.
- **Wrong date filtering:** convert the column with `pd.to_datetime` first.
- **`SettingWithCopyWarning` / chained assignment problems:** make a copy when needed and update with `.loc`.
- **Unexpected empty filter:** inspect unique values with `.unique()` and check spelling, case, whitespace, and data types.
- **Merge has too many rows:** check duplicate join keys and the intended join type.
- **`TypeError` during arithmetic:** inspect `df.dtypes`; convert strings to numeric where appropriate.
- **Missing-value surprises:** use `isna()` and remember that missing values need special handling.

In [ ]:
# Handy debugging checklist
print("Columns:", df.columns.tolist())
print("\nTypes:\n", df.dtypes)
print("\nShape:", df.shape)
print("\nMissing:\n", df.isna().sum())
print("\nUnique day values:", df["day"].astype("string").unique())

## 23. SQL-to-pandas quick reference


| SQL idea | pandas equivalent |
|---|---|
| `SELECT col1, col2` | `df[['col1', 'col2']]` |
| `WHERE tip > 3` | `df[df['tip'] > 3]` |
| `ORDER BY tip DESC` | `df.sort_values('tip', ascending=False)` |
| `GROUP BY day` | `df.groupby('day')` |
| `COUNT(*)` | `df.groupby(...).size()` or `value_counts()` |
| `JOIN` | `pd.merge(left, right, on='key', how='left')` |
| `DISTINCT` | `df['col'].unique()` or `df[['col']].drop_duplicates()` |
| `CASE WHEN` | `np.where()` or `np.select()` |
| `LIMIT 10` | `df.head(10)` |


## 24. Guided mini-project: restaurant tipping analysis

Use pandas to answer these questions. Run the starter cell, then write your own code in the empty cells.

In [ ]:
# Start from a clean copy of the raw dataset
raw = pd.read_csv(DATA_URL)
work = raw.copy()
work.head()

### Task 1 — Inspect
- How many rows and columns are there?
- Which columns contain missing values?
- What are the data types?

In [ ]:
# Your code here

### Task 2 — Clean
- Check for duplicate rows.
- Ensure `total_bill` and `tip` are numeric.
- Create a clean copy rather than changing `raw`.

In [ ]:
# Your code here

### Task 3 — Feature engineering
- Create `tip_pct` as tip / total_bill * 100.
- Create a `bill_group` column with Small (<15), Medium (15 to <30), and Large (>=30).

In [ ]:
# Your code here

### Task 4 — Grouped analysis
- Calculate number of visits, average bill, average tip, and average tip percentage by day.
- Which day has the highest average tip percentage?

In [ ]:
# Your code here

### Task 5 — Compare groups
- Compare average tips for smokers and non-smokers.
- Compare lunch and dinner average bills.
- Use `groupby` and `agg`.

In [ ]:
# Your code here

### Task 6 — Visualise
- Plot average tip percentage by day.
- Plot total bill against tip.
- Write one or two sentences describing what you observe.

In [ ]:
# Your code here

## 25. Practice exercises (try before checking answers)

1. Show the first 12 rows.
2. Select only `total_bill`, `tip`, and `day`.
3. Find rows where the bill is over 25 and the tip is at least 3.
4. Find the 10 largest bills.
5. Calculate the median bill by day.
6. Count visits by day and time.
7. Find the average tip for each party size.
8. Find rows with a tip percentage above 20%.
9. Create a table of average bill by day and meal time.
10. Export the cleaned data to CSV.

### Answer key

In [ ]:
# 1
raw.head(12)

In [ ]:
# 2
raw[["total_bill", "tip", "day"]]

In [ ]:
# 3
raw[(raw["total_bill"] > 25) & (raw["tip"] >= 3)]

In [ ]:
# 4
raw.nlargest(10, "total_bill")

In [ ]:
# 5
raw.groupby("day")["total_bill"].median().sort_values(ascending=False)

In [ ]:
# 6
raw.groupby(["day", "time"]).size().unstack(fill_value=0)

In [ ]:
# 7
raw.groupby("size")["tip"].mean().sort_values(ascending=False)

In [ ]:
# 8
raw.assign(tip_pct=raw["tip"] / raw["total_bill"] * 100).query("tip_pct > 20")

In [ ]:
# 9
raw.pivot_table(index="day", columns="time", values="total_bill", aggfunc="mean")

In [ ]:
# 10
raw.to_csv("tips_practice_export.csv", index=False)

## 26. Essential pandas cheat sheet


| Goal | Command |
|---|---|
| Load CSV | `pd.read_csv(path_or_url)` |
| Save CSV | `df.to_csv(path, index=False)` |
| Preview | `df.head()` / `df.tail()` |
| Dimensions | `df.shape` |
| Column names | `df.columns` |
| Types and null counts | `df.info()` |
| Summary stats | `df.describe()` |
| Select column(s) | `df['col']` / `df[['a', 'b']]` |
| Select by labels/positions | `df.loc[...]` / `df.iloc[...]` |
| Filter | `df[df['col'] > value]` |
| Sort | `df.sort_values('col')` |
| Unique values | `df['col'].unique()` |
| Frequency counts | `df['col'].value_counts()` |
| Missing values | `df.isna().sum()` |
| Drop missing rows | `df.dropna()` |
| Fill missing values | `df['col'].fillna(value)` |
| Duplicates | `df.duplicated().sum()` / `df.drop_duplicates()` |
| New column | `df['new'] = ...` |
| Group summary | `df.groupby('group')['value'].mean()` |
| Multiple summaries | `df.groupby('group').agg(...)` |
| Join tables | `pd.merge(a, b, on='key', how='left')` |
| Stack tables | `pd.concat([a, b])` |
| Pivot summary | `pd.pivot_table(...)` |
| Convert dates | `pd.to_datetime(df['date'], errors='coerce')` |
| Text methods | `df['text'].str.strip()` |
| Correlation | `df.select_dtypes('number').corr()` |
| Plot | `df.plot(...)` |


## 27. Official resources for continued learning

- [Pandas Getting Started](https://pandas.pydata.org/docs/getting_started/)
- [Pandas User Guide](https://pandas.pydata.org/docs/user_guide/)
- [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Pandas API reference](https://pandas.pydata.org/docs/reference/index.html)
- [Practice dataset CSV](https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv)

**Suggested learning order:** inspect → select/filter → clean → transform → group/aggregate → merge → reshape → dates/text → visualise → mini-project.